# STEP 1 - INSTALL LIBRARIES


In [ ]:
!nvidia-smi
!pip install -q transformers datasets peft accelerate
!pip install -q fastapi uvicorn pyngrok nest_asyncio
!pip install -q detoxify

# STEP 2 - IMPORT LIBRARIES

In [ ]:
import torch
import time
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)
from peft import LoraConfig, get_peft_model
from detoxify import Detoxify

# STEP 3 - LOAD DATASET

In [ ]:
ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')

# STEP 4 - REDUCE DATASET SIZE

In [ ]:
small_ds = ds['train'].select(range(5000))
print(small_ds[0])

# STEP 5 - FORMAT DATASET

In [ ]:
def format_data(example):
    text = f"""
### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""
    return {'text': text}

small_ds = small_ds.map(format_data)

# STEP 6 - LOAD MODEL

In [ ]:
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'
)
print('Model Loaded')

# STEP 7 - APPLY LORA

In [ ]:
!pip install --upgrade torchao

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, peft_config)
print('LoRA Applied')

# STEP 8 - TOKENIZE DATASET

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )

tokenized_ds = small_ds.map(tokenize_function)

# STEP 9 - DATA COLLATOR

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# STEP 10 - TRAINING ARGUMENTS

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=20
)

# STEP 11 - CREATE TRAINER

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator
)

# STEP 12 - TRAIN MODEL

In [ ]:
trainer.train()

# STEP 13 - SAVE MODEL

In [ ]:
model.save_pretrained('customer_support_model')
tokenizer.save_pretrained('customer_support_model')
print('Model Saved')

# STEP 14 - LOAD PIPELINE

In [ ]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer
)

# STEP 15 - TEST MODEL

In [ ]:
query = 'I want refund because my product arrived damaged.'
result = pipe(query, max_new_tokens=100)
print(result[0]['generated_text'])

# STEP 16 - TOXICITY FILTERING

In [ ]:
def safe_response(text):
    result = Detoxify('original').predict(text)
    if result['toxicity'] > 0.7:
        return 'Unsafe response blocked.'
    return text

# STEP 17 - PROMPT GUARDRAILS

In [ ]:
blocked_words = [
    'ignore previous instructions',
    'hack',
    'reveal system prompt',
    'password'
]

def validate_prompt(prompt):
    for word in blocked_words:
        if word.lower() in prompt.lower():
            return False
    return True

# STEP 18 - FINAL CHATBOT

In [ ]:
def chatbot(query):
    if not validate_prompt(query):
        return 'Unsafe prompt blocked.'
    result = pipe(query, max_new_tokens=100)
    output = result[0]['generated_text']
    return safe_response(output)

# STEP 19 - TEST CHATBOT

In [ ]:
print(chatbot('I want refund because my package is damaged.'))

# STEP 20 - BENCHMARKING

In [ ]:
start = time.time()
response = chatbot('I need refund')
end = time.time()
print('Latency:', end - start)
memory = torch.cuda.memory_allocated() / 1024**3
print('GPU Memory:', memory, 'GB')

# STEP 21 - CREATE FASTAPI FILE

In [ ]:
fastapi_code = '''
from fastapi import FastAPI
from transformers import pipeline
from pydantic import BaseModel

app = FastAPI()

class ChatRequest(BaseModel):
    query: str

@app.get(\"/\")
def home():
    return {\"message\": \"Customer Support API Running\"}

@app.post(\"/chat\")
def chat(request: ChatRequest):
    return {\"response\": request.query}
'''

with open('app.py', 'w') as f:
    f.write(fastapi_code)
    
print('FastAPI app created')

# STEP 22 - NGROK SETUP

In [ ]:
from pyngrok import ngrok

# Note: Replace with your own ngrok auth token
# ngrok.set_auth_token(\"YOUR_NGROK_TOKEN\")
# public_url = ngrok.connect(8000)
# print(public_url)

# STEP 23 - UPLOAD MODEL TO HUGGING FACE

In [ ]:
# Uncomment and run after logging in with huggingface_hub
# from huggingface_hub import notebook_login
# notebook_login()

# model.push_to_hub('customer-support-llm')
# tokenizer.push_to_hub('customer-support-llm')
# print('Project Completed Successfully')